# Modeling PoC

Proof of concept: preprocessing → feature engineering → pipeline → hyperparameter tuning (GridSearchCV) → evaluation, for Ridge and Decision Tree regressors on the Ames house price data. Decisions below (NA handling, MSSubClass as categorical, outlier rows, log target) come straight from `notebook.ipynb`'s EDA — see that notebook for the reasoning.

Focus is the end-to-end flow, not maximizing accuracy.


## 1. Load & clean

Same fixes as the EDA: the `NA`/`None` load bug, NA-as-category fill, `MSSubClass` cast, outlier drop.


In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
NA_IS_CATEGORY = [
    "Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "PoolQC", "Fence", "MiscFeature",
]
OUTLIER_IDS = [524, 1299]

df = pd.read_csv("data/train.csv", na_values=["NA"], keep_default_na=False)
df = df[~df["Id"].isin(OUTLIER_IDS)].reset_index(drop=True)
df[NA_IS_CATEGORY] = df[NA_IS_CATEGORY].fillna("None")
df["MSSubClass"] = df["MSSubClass"].astype(str)

print("rows after outlier drop:", len(df))
df.isna().sum().pipe(lambda s: s[s > 0])


rows after outlier drop: 1458


LotFrontage    259
MasVnrType       8
MasVnrArea       8
Electrical       1
GarageYrBlt     81
dtype: int64

## 2. Feature engineering & split

Two simple engineered features to demonstrate the step exists (not an exhaustive search): total finished square footage and house age at sale. Target is `log1p(SalePrice)` (see EDA step 2 — the raw target is heavily right-skewed).


In [ ]:
df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

y = np.log1p(df["SalePrice"])
X = df.drop(columns=["Id", "SalePrice"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("train:", X_train.shape, " test:", X_test.shape)


train: (1166, 81)  test: (292, 81)


## 3. Preprocessing pipeline

Numeric → median impute (+ scale, needed for Ridge, harmless for the tree). Categorical → most-frequent impute (covers the 5 genuinely-missing columns; the 14 NA-is-category columns are already filled) + one-hot encode with `handle_unknown="ignore"` (EDA step 10 showed `test.csv` has an `MSSubClass` level absent from `train.csv` — the same can happen with any future dataset).


In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()
print(len(numeric_features), "numeric,", len(categorical_features), "categorical features")

numeric_transformer = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_transformer = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])


37 numeric, 44 categorical features


## 4. Model pipelines + hyperparameter tuning

Plain `LinearRegression` has no hyperparameter to tune, so we use `Ridge` (L2-regularized linear regression) to give `GridSearchCV` something real to search over — the pipeline mechanics are what matter for this PoC, not the specific linear variant. Small grids, kept fast for a live demo.


In [ ]:
ridge_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", Ridge()),
])
ridge_grid = {"model__alpha": [0.1, 1.0, 10.0, 50.0, 100.0]}

tree_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42)),
])
tree_grid = {
    "model__max_depth": [4, 6, 8, 10, None],
    "model__min_samples_leaf": [1, 5, 10],
}

models = {
    "Ridge": (ridge_pipeline, ridge_grid),
    "DecisionTree": (tree_pipeline, tree_grid),
}


In [5]:
fitted = {}
for name, (pipeline, grid) in models.items():
    search = GridSearchCV(pipeline, grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(X_train, y_train)
    fitted[name] = search
    print(f"{name}: best params = {search.best_params_}, best CV RMSE (log) = {-search.best_score_:.4f}")


Ridge: best params = {'model__alpha': 10.0}, best CV RMSE (log) = 0.1139


DecisionTree: best params = {'model__max_depth': 6, 'model__min_samples_leaf': 10}, best CV RMSE (log) = 0.1737


## 5. Evaluate on held-out test split

RMSE on the log scale (the Kaggle metric, per PLAN.md) plus R², for the tuned model of each type.


In [ ]:
results = []
for name, search in fitted.items():
    pred = search.predict(X_test)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2 = r2_score(y_test, pred)
    results.append({"model": name, "test_rmse_log": rmse, "test_r2": r2, "best_params": search.best_params_})

pd.DataFrame(results)


,model,test_rmse_log,test_r2,best_params
0,Ridge,0.122758,0.910608,{'model__alpha': 10.0}
1,DecisionTree,0.197259,0.769179,"{'model__max_depth': 6, 'model__min_samples_le..."


## 6. Persist trained pipelines

Refit each tuned model on the *full* cleaned dataset (train+test) before saving, so the Streamlit demo uses every available row, not just the 80% training split. Also save training-data medians/modes so the app can default any input the user doesn't provide.


In [ ]:
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

for name, search in fitted.items():
    best_pipeline = search.best_estimator_
    best_pipeline.fit(X, y)  # refit on all cleaned rows
    joblib.dump(best_pipeline, models_dir / f"{name.lower()}_pipeline.joblib")

# Defaults so the Streamlit app can fill in any column the user doesn't provide
defaults = {}
for col in numeric_features:
    defaults[col] = float(X[col].median())
for col in categorical_features:
    defaults[col] = X[col].mode().iloc[0]
with open(models_dir / "defaults.json", "w") as f:
    json.dump(defaults, f, indent=2)

# Category options for Streamlit selectboxes
options = {col: sorted(X[col].dropna().unique().tolist()) for col in categorical_features}
with open(models_dir / "categorical_options.json", "w") as f:
    json.dump(options, f, indent=2)

print("Saved:", [p.name for p in models_dir.iterdir()])


Saved: ['defaults.json', 'categorical_options.json', 'decisiontree_pipeline.joblib', 'ridge_pipeline.joblib']
